# Token Budget Manager — Cost-Controlled Research Agent
## Cap the Cost — Token-Aware Research Agent
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/63-token-budget-manager/token_budget_workbook.ipynb)

LLM calls cost money — and uncapped agents can burn through a budget fast. This workshop builds a **token budget manager**: a LangGraph research agent that counts tokens with `tiktoken` at every step, enforces a hard cap, and short-circuits gracefully when the budget is exceeded. By the end you will understand *how* to track prompt + completion tokens per node and *how* a conditional edge lets you exit early rather than run to completion and overspend.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why token budgets matter and how tiktoken works |
| 2 | **Token Counting** — `count_tokens()` with tiktoken |
| 3 | **Budget Check** — Track usage per step, detect overage |
| 4 | **Conditional Exit** — Route to `synthesize` or short-circuit via `END` |
| 5 | **Full Pipeline** — research_step → check_budget → [synthesize \| END] |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "tiktoken", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Why Token Budgets Matter

| Without a budget | With a budget |
|---|---|
| Agent runs all steps regardless of cost | Hard cap terminates early |
| Overspend discovered at invoice time | Overspend caught in real-time |
| No partial result on timeout | Partial notes returned gracefully |

**tiktoken** is OpenAI's tokenizer — it counts tokens *before* the API call (prompt tokens) and *after* (completion tokens). Both count toward billing.

In [ ]:
import tiktoken

# 800 tokens is intentionally tight — raise to 3000+ for real research tasks
BUDGET_LIMIT = 800
RESEARCH_TOPIC = "Explain the history and key contributions of neural networks"

RESEARCH_STEPS = [
    "Step 1: Briefly describe the perceptron (1958) and its limitations.",
    "Step 2: Explain the backpropagation breakthrough (1986).",
    "Step 3: Describe the deep learning revolution (2012, AlexNet).",
    "Step 4: Summarize the transformer architecture (2017, Attention is All You Need).",
]

_enc = tiktoken.encoding_for_model("gpt-4o")

# tiktoken counts tokens client-side without an API call — avoids billing surprises
def count_tokens(text: str) -> int:
    return len(_enc.encode(text))

print(f"Budget: {BUDGET_LIMIT} tokens  |  Steps: {len(RESEARCH_STEPS)}")
print(f"Step 1 prompt size: {count_tokens(RESEARCH_STEPS[0])} tokens")

---
## Part 2 — Token Counting with tiktoken

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# temperature=0 for deterministic responses — same prompt always returns the same tokens
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Count a single step manually
step_prompt = RESEARCH_STEPS[0]
system_msg = f"You are a researcher summarizing: {RESEARCH_TOPIC}"

prompt_tokens = count_tokens(step_prompt)
response = llm.invoke([SystemMessage(content=system_msg), HumanMessage(content=step_prompt)])
completion_tokens = count_tokens(response.content)

print(f"Prompt tokens:     {prompt_tokens}")
print(f"Completion tokens: {completion_tokens}")
print(f"Step total:        {prompt_tokens + completion_tokens}")
print(f"Budget remaining:  {BUDGET_LIMIT - (prompt_tokens + completion_tokens)} / {BUDGET_LIMIT}")
print(f"\nResponse: {response.content[:200]}...")

---
## Part 3 — Budget Check Node

After each research step, `check_budget` compares `tokens_used` to `BUDGET_LIMIT`. If exceeded, it sets `budget_exceeded=True` and populates `final_answer` with a partial result so nothing is lost.

In [ ]:
# Simulate budget tracking across steps
running_total = 0
notes = []

for i, step in enumerate(RESEARCH_STEPS):
    if running_total >= BUDGET_LIMIT:
        print(f"  ⛔ Step {i+1} SKIPPED — budget exceeded ({running_total}/{BUDGET_LIMIT})")
        continue
    p = count_tokens(step)
    r = llm.invoke([SystemMessage(content=system_msg), HumanMessage(content=step)])
    c = count_tokens(r.content)
    step_total = p + c
    running_total += step_total
    notes.append(r.content.strip())
    status = "⛔ OVER" if running_total >= BUDGET_LIMIT else "✓"
    print(f"  Step {i+1}: +{step_total} tokens → total={running_total}/{BUDGET_LIMIT} {status}")

print(f"\nCompleted {len(notes)}/{len(RESEARCH_STEPS)} steps  |  Final usage: {running_total} tokens")

---
## Part 4 — Conditional Exit

```python
def route(state) -> Literal["research_step", "synthesize", END]:
    if state["budget_exceeded"]:  return END          # hard stop
    if state["current_step"] >= len(state["steps"]): return "synthesize"  # done
    return "research_step"                             # keep going
```

Three outcomes: budget hit → `END` (partial result), all steps done → `synthesize`, otherwise → next step.

In [ ]:
from langgraph.graph import END

# Return value of route() maps directly to a node name or END — LangGraph resolves the edge
def route(state):
    if state["budget_exceeded"]:                     return "__end__"
    if state["current_step"] >= len(state["steps"]): return "synthesize"
    return "research_step"

# Demonstrate routing decisions
scenarios = [
    {"budget_exceeded": True,  "current_step": 2, "steps": RESEARCH_STEPS},
    {"budget_exceeded": False, "current_step": 4, "steps": RESEARCH_STEPS},
    {"budget_exceeded": False, "current_step": 1, "steps": RESEARCH_STEPS},
]
for s in scenarios:
    print(f"exceeded={s['budget_exceeded']}, step={s['current_step']}/{len(s['steps'])} → {route(s)}")

---
## Part 5 — Full LangGraph Pipeline

```
START → research_step → check_budget → [research_step (loop) | synthesize | END]
```

In [ ]:
from typing import Literal, TypedDict
from langgraph.graph import START, StateGraph

# TypedDict lets LangGraph validate state keys at graph-build time with no runtime overhead
class BudgetState(TypedDict):
    topic: str; steps: list; current_step: int; notes: list
    tokens_used: int; token_log: list; budget_exceeded: bool; final_answer: str

system_msg_full = SystemMessage(content=f"You are a researcher summarizing: {RESEARCH_TOPIC}")

def research_step_node(state):
    idx = state["current_step"]
    if idx >= len(state["steps"]): return {}
    step = state["steps"][idx]
    p = count_tokens(step)
    r = llm.invoke([system_msg_full, HumanMessage(content=step)])
    c = count_tokens(r.content)
    new_total = state["tokens_used"] + p + c
    log = {"step": idx+1, "prompt": p, "completion": c, "total": p+c}
    return {"notes": state["notes"] + [r.content.strip()], "current_step": idx+1,
            "tokens_used": new_total, "token_log": state["token_log"] + [log]}

def check_budget_node(state):
# Nodes return partial dicts — LangGraph merges them into state; unmentioned keys are unchanged
    if state["tokens_used"] >= BUDGET_LIMIT:
        partial = "\n\n".join(f"Step {i+1}: {n}" for i, n in enumerate(state["notes"]))
        return {"budget_exceeded": True,
                "final_answer": f"[BUDGET EXCEEDED at {state['tokens_used']} tokens]\n\n{partial}"}
    return {"budget_exceeded": False}

def route_node(state) -> Literal["research_step", "synthesize", "__end__"]:
    if state["budget_exceeded"]: return END
    if state["current_step"] >= len(state["steps"]): return "synthesize"
    return "research_step"

def synthesize_node(state):
    combined = "\n\n".join(f"Step {i+1}: {n}" for i, n in enumerate(state["notes"]))
    ans = llm.invoke([system_msg_full,
        HumanMessage(content=f"Synthesize these research notes into a coherent summary:\n{combined}")])
    return {"final_answer": ans.content.strip()}

g = StateGraph(BudgetState)
for name, fn in [("research_step", research_step_node), ("check_budget", check_budget_node), ("synthesize", synthesize_node)]:
    g.add_node(name, fn)
g.add_edge(START, "research_step")
g.add_edge("research_step", "check_budget")
g.add_conditional_edges("check_budget", route_node)
# add_conditional_edges: route_node's return value is matched against node names (or END)
g.add_edge("synthesize", END)
app = g.compile()
# compile() locks the graph topology — after this, nodes and edges cannot be added or changed

init = {"topic": RESEARCH_TOPIC, "steps": RESEARCH_STEPS, "current_step": 0,
        "notes": [], "tokens_used": 0, "token_log": [], "budget_exceeded": False, "final_answer": ""}
result = app.invoke(init)

print(f"Steps completed: {result['current_step']}/{len(RESEARCH_STEPS)}")
print(f"Total tokens used: {result['tokens_used']} / {BUDGET_LIMIT}")
print(f"Budget exceeded: {result['budget_exceeded']}")
print(f"\nToken log:")
for entry in result["token_log"]:
    print(f"  Step {entry['step']}: prompt={entry['prompt']}, completion={entry['completion']}, total={entry['total']}")
print(f"\nFinal answer:\n{result['final_answer'][:400]}")

---
### Exercise 1 — Tighten the Budget
Set `BUDGET_LIMIT = 200`. How many steps complete before the hard stop? What does the partial result look like?

### Exercise 2 — Per-Step Budget
Instead of a global budget, enforce a *per-step* cap of 150 tokens. If a single step exceeds it, skip that step and continue (don't exit). Modify `check_budget_node` to implement this.

In [ ]:
# Exercise 1 — tight budget
BUDGET_TIGHT = 200
tight_init = {**init, "steps": RESEARCH_STEPS}
# Swap BUDGET_LIMIT in check_budget_node to BUDGET_TIGHT and re-run
print(f"With BUDGET_LIMIT={BUDGET_TIGHT}, expect early exit after ~1 step")
for i, entry in enumerate(result["token_log"]):
    cumulative = sum(e["total"] for e in result["token_log"][:i+1])
    would_exit = cumulative >= BUDGET_TIGHT
    print(f"  Step {entry['step']}: cumulative={cumulative} → exit={would_exit}")

# Exercise 2 — per-step cap
PER_STEP_CAP = 150
for entry in result["token_log"]:
    skipped = entry["total"] > PER_STEP_CAP
    print(f"Step {entry['step']}: {entry['total']} tokens → {'SKIP' if skipped else 'KEEP'}")

---
*Workshop complete. Next: example 64 — Pydantic AI.*